# Iris Flower Classification - Entraînement du Modèle

Ce notebook démontre le pipeline complet de Machine Learning :
1. Chargement et exploration du dataset Iris
2. Prétraitement des données
3. Entraînement d'un modèle Random Forest
4. Évaluation du modèle
5. Export du modèle pour l'API FastAPI

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import json
import os

## 1. Chargement du Dataset Iris

Le dataset Iris contient 150 échantillons de 3 espèces de fleurs d'iris :
- **Setosa**
- **Versicolor**
- **Virginica**

Chaque échantillon possède 4 caractéristiques (features) :
- Longueur du sépale (cm)
- Largeur du sépale (cm)
- Longueur du pétale (cm)
- Largeur du pétale (cm)

In [ ]:
iris = load_iris()

df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Dimensions du dataset: {df.shape}")
print(f"\nColonnes: {list(df.columns)}")
print(f"\nEspèces: {list(df['species'].unique())}")
df.head(10)

## 2. Exploration des Données

In [ ]:
print("=== Statistiques descriptives ===")
df.describe()

In [ ]:
print("=== Distribution par espèce ===")
print(df['species'].value_counts())
print(f"\n=== Valeurs manquantes ===")
print(df.isnull().sum())

## 3. Préparation des Données

In [ ]:
X = df[iris.feature_names].values
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille de l'ensemble d'entraînement: {X_train.shape[0]}")
print(f"Taille de l'ensemble de test: {X_test.shape[0]}")

## 4. Entraînement du Modèle Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)
print("Modèle entraîné avec succès !")

## 5. Évaluation du Modèle

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"\n=== Rapport de classification ===")
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print(f"\n=== Matrice de confusion ===")
print(confusion_matrix(y_test, y_pred))

In [ ]:
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': iris.feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("=== Importance des features ===")
feature_importance_df

## 6. Export du Modèle

On sauvegarde le modèle entraîné et les métadonnées pour le backend FastAPI.

In [ ]:
model_dir = os.path.join('..', 'backend', 'models')
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, 'iris_model.joblib')
joblib.dump(model, model_path)
print(f"Modèle sauvegardé: {model_path}")

metadata = {
    'model_type': 'RandomForestClassifier',
    'n_estimators': 100,
    'max_depth': 5,
    'accuracy': float(accuracy),
    'feature_names': list(iris.feature_names),
    'target_names': list(iris.target_names),
    'feature_importances': {name: float(imp) for name, imp in zip(iris.feature_names, importances)},
    'training_samples': int(X_train.shape[0]),
    'test_samples': int(X_test.shape[0])
}

metadata_path = os.path.join(model_dir, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Métadonnées sauvegardées: {metadata_path}")

## 7. Test de Prédiction

Vérifions que le modèle sauvegardé fonctionne correctement.

In [ ]:
loaded_model = joblib.load(model_path)

sample = np.array([[5.1, 3.5, 1.4, 0.2]])
prediction = loaded_model.predict(sample)
probabilities = loaded_model.predict_proba(sample)

print(f"Échantillon: {sample[0]}")
print(f"Prédiction: {iris.target_names[prediction[0]]}")
print(f"Probabilités: {dict(zip(iris.target_names, probabilities[0].round(4)))}")
print(f"\n✅ Le modèle est prêt pour le déploiement via FastAPI !")